# 08 - Shared image and Python workbench

Paste these Bash blocks into one Ubuntu terminal. The image contains the common tools used by HDFS, YARN, Hive, Spark and Kafka. Image construction can take several minutes. Source and DAG files remain in your chosen workspace directory.

## Open your workspace

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
source versions.env
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state airflow/dags
```



## Check build capacity

```bash
df -h "$WORKSPACE_HOME"
docker info --format "{{.ServerVersion}}"
```

Allow at least 50 GiB free for image layers and the temporary import archive.

## Fetch verified connectors

```bash
if ! docker image inspect "$TRAINING_IMAGE" >/dev/null 2>&1; then
  python3 image/fetch.py
fi
```

Every connector and the JVM metrics agent is checked against artifacts.lock. Generated downloads stay out of Git.

## Build the shared image

```bash
if docker image inspect "$TRAINING_IMAGE" >/dev/null 2>&1; then
  printf "Using existing image %s\n" "$TRAINING_IMAGE"
else
  docker build --no-cache -t "$TRAINING_IMAGE" image/
fi
```

Version 1.2.0 includes the pinned JMX metrics agent. Nothing is installed into your Ubuntu Python interpreter.
An existing validated image is reused. To deliberately rebuild, remove that image tag first; ordinary cluster recreation only needs the export/import steps below.


## Export the image

```bash
docker save -o .state/training-image.tar "$TRAINING_IMAGE"
```



## Import into the node runtime

```bash
docker exec -i bigdata ctr --namespace k8s.io images import - < .state/training-image.tar
```

Host Docker and the node containerd runtime have separate image stores. This Docker exec targets the Minikube node, not an application pod.

## Remove the temporary archive

```bash
rm .state/training-image.tar
```



## Start the workbench

```bash
bash start-components.sh jupyter
```

The helper applies telemetry configuration, starts the workspace mount and waits for Jupyter. It shares the complete project at /workspace.

## Check the shared directory

```bash
kubectl exec jupyter-0 -c jupyter -- ls /workspace/00-setup /workspace/notebooks /workspace/python /workspace/pyspark /workspace/airflow/dags
```



## Check the Python environment

```bash
kubectl exec jupyter-0 -c jupyter -- python -c 'import sys,pyspark; print(sys.prefix,pyspark.__version__); assert sys.prefix=="/opt/venv"; assert pyspark.__version__=="3.5.9"'
```



## Open JupyterLab

Run this in a second terminal and keep it open. Continue the setup in your original terminal.

```bash
kubectl port-forward --address=127.0.0.1 service/jupyter 8889:8888
```

Keep this command open in a second terminal. In the Guacamole desktop browser open http://127.0.0.1:8889 and enter the lab token team1234. Open notebooks/ for exercises; no laptop tunnel is needed.

## Stop the workbench

```bash
bash stop-components.sh jupyter
```

Files remain in the host workspace.

## Resume the workbench

```bash
bash start-components.sh jupyter
```



## Check the retained host files

```bash
kubectl exec jupyter-0 -c jupyter -- test -f /workspace/versions.env
kubectl exec jupyter-0 -c jupyter -- test -f /workspace/00-setup/01-minikube-cluster.ipynb
```

The same host project is attached after replacing the workbench pod.